### CasePilot

In [ ]:
import os
print(os.listdir())

['.config', '__pycache__', 'token_primary.json', 'generate_dummy_cases.py', 'structured_cases_progress.json', '.ipynb_checkpoints', 'credentials.json', 'casepilot_structured_cases.csv', 'token_support.json', 'casepilot_scored_cases.csv', 'gemini_structure.py', 'export_results.py', 'gmail_ingest.py', 'sample_data']


In [ ]:
import importlib; importlib.reload(gdc)

<module 'generate_dummy_cases' from '/content/generate_dummy_cases.py'>

In [ ]:
!pip install google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client gspread google-generativeai --quiet

In [ ]:
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build

SCOPES = [
    "https://www.googleapis.com/auth/gmail.readonly",
    "https://www.googleapis.com/auth/gmail.send",
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]

creds = Credentials.from_authorized_user_file("token_support.json", SCOPES)
service = build("gmail", "v1", credentials=creds)

sent = 0
for i in range(1, 101):
    email_data = gdc.generate_dummy_email(i)
    try:
        gdc.send_email(service, "casepilot.demo@gmail.com", email_data["subject"], email_data["body"])
        sent += 1
        if sent % 10 == 0:
            print(f"Sent {sent}/100...")
    except Exception as e:
        print(f"Failed on #{i}: {e}")

print(f"Done. Sent {sent}/100 dummy case emails.")

Sent 10/100...
Sent 20/100...
Sent 30/100...
Sent 40/100...
Sent 50/100...
Sent 60/100...
Sent 70/100...
Sent 80/100...
Sent 90/100...
Sent 100/100...
Done. Sent 100/100 dummy case emails.


In [ ]:
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY3')
genbrain = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build

In [ ]:
import gmail_ingest
print(gmail_ingest.SCOPES)

['https://www.googleapis.com/auth/gmail.readonly']


In [ ]:
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
import gmail_ingest

creds = Credentials.from_authorized_user_file(
    "token_support.json",
    gmail_ingest.SCOPES
)

service_support = build(
    "gmail",
    "v1",
    credentials=creds
)

cases = gmail_ingest.fetch_case_emails(
    service_support,
    query='subject:"[Case #"',
    max_results=100
)

print(f"Fetched {len(cases)} cases")

Fetched 100 cases


In [ ]:
print(cases)

[{'case_id': '19f317019e616784', 'from': 'CasePilot <casepilot.demo@gmail.com>', 'subject': '[Case #100] Login failing with error code AUTH-{n}', 'date': 'Sun, 5 Jul 2026 01:40:57 -0700', 'snippet': 'Hi Support Team, We&#39;re seeing intermittent 500 Internal Server Error responses from the /v2/reports/100 endpoint since ~09:00 UTC today. Error code: ERR-100. Roughly 1 in 100 requests fail. This is', 'body': "Hi Support Team,\n\nWe're seeing intermittent 500 Internal Server Error responses from the /v2/reports/100 endpoint since ~09:00 UTC today. Error code: ERR-100. Roughly 1 in 100 requests fail. This is blocking our production reporting pipeline — can we get an ETA on a fix or at least a root-cause update within the next 48 hours?\n\nThanks,\nDavid Kim"}, {'case_id': '19f317014c963e6b', 'from': 'CasePilot <casepilot.demo@gmail.com>', 'subject': '[Case #99] Dashboard not loading after latest update', 'date': 'Sun, 5 Jul 2026 03:40:56 -0500', 'snippet': 'Hi Support Team, We&#39;re see

In [ ]:
import json

EXTRACTION_PROMPT = """You are a customer support case classifier and decision-support assistant. Given the email below, extract:
- issue_category (string, one of: billing, technical, access, feature_request, complaint, other)
- sentiment (string: positive, neutral, negative, urgent_negative)
- urgency (integer 1-5, 5 = most urgent)
- key_entities (array of strings: product names, account IDs, error codes mentioned)
- summary (string, one sentence)
- root_cause (string, one sentence: likely underlying cause, not just the symptom)
- recommended_next_action (string, one sentence: concrete next step for the team)
- business_impact (string, one of: low, medium, high, critical)
- confidence (float 0-1: your confidence in this classification)

Respond ONLY with valid JSON matching this schema. No preamble, no markdown fences.

EMAIL:
Subject: {subject}
From: {sender}
Body: {body}
"""

def structure_case(case):
    prompt = EXTRACTION_PROMPT.format(
        subject=case["subject"], sender=case["from"], body=case["body"]
    )
    response = genbrain.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt,
        config={"temperature": 0.2, "response_mime_type": "application/json"},
    )
    try:
        structured = json.loads(response.text)
    except json.JSONDecodeError:
        structured = {"error": "parse_failed", "raw": response.text}

    structured["case_id"] = case["case_id"]
    structured["from"] = case["from"]
    structured["subject"] = case["subject"]
    structured["date"] = case["date"]
    return structured

In [ ]:
import importlib
importlib.reload(gmail_ingest)
cases = gmail_ingest.fetch_case_emails(service_support, query='Subject:"[Case #"', max_results=100)
result = structure_case(cases[0])
import pprint
pprint.pprint(result)

{'business_impact': 'high',
 'case_id': '19f317019e616784',
 'confidence': 0.95,
 'date': 'Sun, 5 Jul 2026 01:40:57 -0700',
 'from': 'CasePilot <casepilot.demo@gmail.com>',
 'issue_category': 'technical',
 'key_entities': ['/v2/reports/100', 'ERR-100'],
 'recommended_next_action': 'Escalate to the backend engineering team to '
                            'investigate server logs for the /v2/reports/100 '
                            'endpoint and provide the client with an update.',
 'root_cause': 'A potential server-side instability or resource contention '
               'issue affecting the /v2/reports/100 endpoint.',
 'sentiment': 'urgent_negative',
 'subject': '[Case #100] Login failing with error code AUTH-{n}',
 'summary': 'The client is experiencing intermittent 500 Internal Server '
            'Errors on the reporting endpoint which is disrupting their '
            'production pipeline.',
 'urgency': 4}


In [ ]:
import gemini_structure
import importlib
importlib.reload(gemini_structure)

<module 'gemini_structure' from '/content/gemini_structure.py'>

In [ ]:
from google.colab import userdata
from google import genai

GEMINI_API_KEY3 = userdata.get('GEMINI_API_KEY3')  # whatever you named the new secret
genbrain3 = genai.Client(api_key=GEMINI_API_KEY3)

import gemini_structure
import importlib
importlib.reload(gemini_structure)

structured_cases = gemini_structure.structure_all_cases(genbrain3, cases)

Processed 10/100 (saved to structured_cases_progress.json)
Processed 20/100 (saved to structured_cases_progress.json)
Processed 30/100 (saved to structured_cases_progress.json)
Processed 40/100 (saved to structured_cases_progress.json)
Processed 50/100 (saved to structured_cases_progress.json)
Processed 60/100 (saved to structured_cases_progress.json)
Processed 70/100 (saved to structured_cases_progress.json)
Processed 80/100 (saved to structured_cases_progress.json)
Processed 90/100 (saved to structured_cases_progress.json)
Processed 100/100 (saved to structured_cases_progress.json)
Done. 100/100 succeeded, 0 failed.


In [ ]:
import export_results
importlib.reload(export_results)

successes, failures = export_results.load_and_split()
df = export_results.export_successes_to_csv(successes)
print(df.shape)
df.head()

Total: 100 | Successful: 100 | Failed: 0
Exported 100 cases to casepilot_structured_cases.csv
(100, 13)


,case_id,subject,from,date,issue_category,sentiment,urgency,business_impact,confidence,root_cause,recommended_next_action,summary,key_entities
0,19f317019e616784,[Case #100] Login failing with error code AUTH...,CasePilot <casepilot.demo@gmail.com>,"Sun, 5 Jul 2026 01:40:57 -0700",technical,urgent_negative,4,high,0.95,Potential instability or resource exhaustion w...,Escalate to the engineering team to investigat...,The client is experiencing intermittent 500 In...,"[/v2/reports/100, ERR-100]"
1,19f317014c963e6b,[Case #99] Dashboard not loading after latest ...,CasePilot <casepilot.demo@gmail.com>,"Sun, 5 Jul 2026 03:40:56 -0500",technical,urgent_negative,4,high,0.98,A potential regression or service instability ...,Escalate to the engineering team to investigat...,The user is reporting intermittent 500 Interna...,"[/v2/reports/99, ERR-99, Case #99]"
2,19f31700ee7ed3dd,[Case #98] This is the third time I'm reportin...,CasePilot <casepilot.demo@gmail.com>,"Sun, 5 Jul 2026 01:40:55 -0700",complaint,urgent_negative,5,high,0.98,A failure in the support ticketing workflow or...,Immediately escalate the ticket to a senior su...,The customer is expressing frustration regardi...,"[ERR-98, ACC-98]"
3,19f31700cbe226df,[Case #97] Refund not received after cancellation,CasePilot <casepilot.demo@gmail.com>,"Sun, 5 Jul 2026 03:40:54 -0500",billing,negative,3,medium,0.98,A delay in the internal financial processing o...,Verify the status of the refund in the payment...,The customer is requesting an update on a $499...,"[CANC-97, ACC-97]"
4,19f31700be54973d,[Case #96] Refund not received after cancellation,CasePilot <casepilot.demo@gmail.com>,"Sun, 5 Jul 2026 01:40:54 -0700",billing,negative,3,medium,0.98,A delay in the internal financial processing o...,Verify the status of refund request CANC-96 in...,The customer is requesting an update on a $129...,"[ACC-96, CANC-96]"


In [ ]:
##Scoring on the basis of what the gemini has interpreted on our based data

In [ ]:
import scoring

scored_df = scoring.load_score_and_save()
scored_df.head(10)

Scored 100 cases -> casepilot_scored_cases.csv
escalation_risk
medium    43
high      42
low       15
Name: count, dtype: int64


,case_id,subject,from,date,issue_category,sentiment,urgency,business_impact,confidence,root_cause,recommended_next_action,summary,key_entities,repeat_complaint_count,health_score,escalation_risk
5,19f3170069e67e9e,[Case #95] This is the third time I'm reportin...,CasePilot <casepilot.demo@gmail.com>,"Sun, 5 Jul 2026 01:40:53 -0700",technical,urgent_negative,5,critical,0.98,Persistent integration failure between the CRM...,Escalate the ticket to a senior technical engi...,The customer is reporting a recurring CRM sync...,"['ERR-95', 'ACC-95']",1,47,high
30,19f316fadfa69582,[Case #70] Very disappointed with recent servi...,CasePilot <casepilot.demo@gmail.com>,"Sun, 5 Jul 2026 01:40:30 -0700",technical,urgent_negative,5,critical,0.98,An unannounced deployment of breaking changes ...,Immediately escalate to the engineering team t...,The user is reporting that recent undocumented...,"['/api/v2', 'ACC-70']",1,47,high
27,19f316fb5783106a,[Case #73] Very disappointed with recent servi...,CasePilot <casepilot.demo@gmail.com>,"Sun, 5 Jul 2026 01:40:32 -0700",technical,urgent_negative,5,critical,0.98,Persistent failure in the integration middlewa...,Escalate the ticket to a senior technical lead...,"The customer is reporting a recurring, unresol...","['ERR-73', 'ACC-73']",1,47,high
19,19f316fcfc030191,[Case #81] Very disappointed with recent servi...,CasePilot <casepilot.demo@gmail.com>,"Sun, 5 Jul 2026 03:40:39 -0500",technical,urgent_negative,5,critical,0.98,An unannounced deployment of breaking changes ...,Escalate to the engineering team to investigat...,The user is reporting that recent undocumented...,"['/api/v2', 'ACC-81']",1,47,high
51,19f316f642500a66,[Case #49] This is the third time I'm reportin...,CasePilot <casepilot.demo@gmail.com>,"Sun, 5 Jul 2026 01:40:11 -0700",technical,urgent_negative,5,critical,0.98,Persistent integration synchronization failure...,Escalate the ticket to a senior technical engi...,The customer is reporting a recurring synchron...,"['ERR-49', 'ACC-49']",1,47,high
45,19f316f776ed0694,[Case #55] Extremely frustrated with lack of r...,CasePilot <casepilot.demo@gmail.com>,"Sun, 5 Jul 2026 04:40:16 -0400",technical,urgent_negative,5,critical,0.98,A recurring technical failure in the CRM integ...,Escalate the ticket to a senior technical engi...,The customer is expressing extreme frustration...,"['ERR-55', 'ACC-55']",1,47,high
41,19f316f8441b1d3e,[Case #59] Data sync broken between environments,CasePilot <casepilot.demo@gmail.com>,"Sun, 5 Jul 2026 04:40:20 -0400",technical,urgent_negative,5,critical,0.98,A potential server-side instability or resourc...,Escalate to the engineering team to investigat...,The user is reporting intermittent 500 Interna...,"['/v2/reports/59', 'ERR-59']",1,47,high
34,19f316f9cc5b8f3d,[Case #66] Extremely frustrated with lack of r...,CasePilot <casepilot.demo@gmail.com>,"Sun, 5 Jul 2026 01:40:26 -0700",technical,urgent_negative,5,critical,0.98,A breaking change was deployed to the producti...,Immediately escalate to the engineering team t...,The customer is reporting that recent undocume...,"['/api/v2', 'ACC-66']",1,47,high
63,19f316f3cff7904b,[Case #37] Very disappointed with recent servi...,CasePilot <casepilot.demo@gmail.com>,"Sun, 5 Jul 2026 01:40:01 -0700",technical,urgent_negative,5,critical,0.98,The engineering team deployed breaking changes...,Escalate to the engineering team to investigat...,The user is reporting that recent undocumented...,"['/api/v2', 'ACC-37']",1,47,high
97,19f316e79cb9f13e,[Case #3] Very disappointed with recent servic...,CasePilot <casepilot.demo@gmail.com>,"Sun, 5 Jul 2026 01:39:11 -0700",technical,urgent_negative,5,critical,0.98,Persistent integration failure between the CRM...,Escalate the ticket to a senior technical engi...,"The customer is reporting a recurring, unresol...","['ERR-3', 'ACC-3']",1,47,high


In [ ]:
scored_df.describe()

,urgency,confidence,repeat_complaint_count,health_score
count,100.000000,100.000000,100.0,100.000000
mean,3.440000,0.975500,1.0,68.220000
std,1.571977,0.010766,0.0,19.008015
min,1.000000,0.950000,1.0,47.000000
25%,2.000000,0.980000,1.0,51.000000
50%,4.000000,0.980000,1.0,57.000000
75%,5.000000,0.980000,1.0,88.000000
max,5.000000,0.980000,1.0,94.000000


### Nvidia Integration

### In the below step, we're upscaling the 100 synthetically generated cases to 100k rows in order to accomadate the same dataset and test the processing time using Nvidia Rapids

In [22]:
import upscale_data

scaled_df = upscale_data.upscale_dataset()
scaled_df.shape

Upscaled 100 base rows -> 100000 rows -> casepilot_scaled_100k.csv
escalation_risk
medium    33456
high      33333
low       33211
Name: count, dtype: int64


(100000, 16)

In [23]:
scaled_df.describe()

,urgency,confidence,repeat_complaint_count,health_score
count,100000.000000,100000.000000,100000.0,100000.00000
mean,3.350220,0.975500,1.0,68.20025
std,1.495175,0.010712,0.0,19.15011
min,1.000000,0.950000,1.0,35.80000
25%,2.000000,0.980000,1.0,50.40000
50%,4.000000,0.980000,1.0,59.60000
75%,5.000000,0.980000,1.0,88.30000
max,5.000000,0.980000,1.0,100.00000


In [29]:
import importlib
importlib.reload(benchmark)

results = benchmark.run_benchmark()

Loaded 100000 rows for benchmarking.

Warming up GPU (excluded from timing - one-time CUDA init cost)...
Warm-up complete.

Running pandas (CPU) version...
  pandas time: 0.0701 seconds

Running cuDF (GPU) version...
  cuDF time: 0.1500 seconds

Speedup: 0.47x faster with cuDF/GPU acceleration
(pandas: 0.0701s -> cuDF: 0.1500s on 100,000 rows)


## Since Nvidia's procees is slow on the size of 100k row's we'll synthetically upsale the data to 2M rows now

In [31]:
import upscale_data
importlib.reload(upscale_data)

scaled_df = upscale_data.upscale_dataset(target_rows=2_000_000, out_path="casepilot_scaled_2m.csv")

Upscaled 100 base rows -> 2000000 rows -> casepilot_scaled_2m.csv
escalation_risk
high      670621
low       666627
medium    662752
Name: count, dtype: int64


In [32]:
scaled_df.describe()

,urgency,confidence,repeat_complaint_count,health_score
count,2.000000e+06,2.000000e+06,2000000.0,2.000000e+06
mean,3.349711e+00,9.755000e-01,1.0,6.821927e+01
std,1.494694e+00,1.071215e-02,0.0,1.914246e+01
min,1.000000e+00,9.500000e-01,1.0,3.170000e+01
25%,2.000000e+00,9.800000e-01,1.0,5.040000e+01
50%,4.000000e+00,9.800000e-01,1.0,5.960000e+01
75%,5.000000e+00,9.800000e-01,1.0,8.840000e+01
max,5.000000e+00,9.800000e-01,1.0,1.000000e+02


In [35]:
importlib.reload(benchmark)
results = benchmark.run_benchmark(csv_path="casepilot_scaled_2m.csv")

Loaded 2000000 rows for benchmarking.

Warming up GPU (excluded from timing - one-time CUDA init cost)...
Warm-up complete.

Running pandas (CPU) version...
  pandas time: 1.8582 seconds

Running cuDF (GPU) version...
  cuDF time: 1.4141 seconds

Speedup: 1.31x faster with cuDF/GPU acceleration
(pandas: 1.8582s -> cuDF: 1.4141s on 2,000,000 rows)
